# XGBoost

In [ ]:
import pandas as pd
import numpy as np

from joblib import Parallel, delayed

import math
import tqdm
import tabulate
import xgboost as xgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.multioutput import MultiOutputClassifier

from pathlib import Path
import warnings

import re

# Nascondo i warning
warnings.filterwarnings('ignore')

# Definisco il percorso dei file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')

# Lista dei csv su cui fare training
datasets = {
    't2_medsam': FILE_PATH / 't2_medsam_masks.csv',
    't2_preprocessed': FILE_PATH / 't2_preprocessed_masks.csv',
    't2_original': FILE_PATH / 't2_original_masks.csv',
    'medsam_dynamic': FILE_PATH / 'medsam_dynamic.csv',
    'preprocessed_dynamic': FILE_PATH / 'preprocessed_dynamic.csv',
    'original_dynamic': FILE_PATH / 'original_dynamic.csv'
}

# Lavoro su singola fold

In [ ]:
def fit_single_fold(train_idx, test_idx, features, target, groups, n_est, max_d, lr, sub, col, mcw, gam, alpha, lam):
  # ========== DEBUGGING: Stampo indici train/test  ==========

  """print("?"*50 + "\nDebug\n" + "?"*50)
  print(f"\nFold {fold} - File: {csv_name}")
  print(f"  Train indice: {train_index[:10]})")
  print(f"  Test indice: {test_index[:10]})")
  print(f"  Train gruppo (Patient IDs): {groups.iloc[train_index].unique()}")
  print(f"  Test gruppo (Patient IDs): {groups.iloc[test_index].unique()}")
  print("?"*100)"""
  # ==========================================================

  # Suddivido i dati in set di training e di test per la fold corrente
  X_train, X_test = features.iloc[train_idx], features.iloc[test_idx]
  y_train, y_test = target.iloc[train_idx], target.iloc[test_idx]

  rf = xgb.XGBClassifier(
        n_estimators= n_est,
        max_depth= max_d,
        learning_rate= lr,
        subsample= sub,
        colsample_bytree= col,
        min_child_weight= mcw,
        gamma= gam,
        reg_alpha= alpha,
        reg_lambda= lam
    )

  multi_output_xgb = MultiOutputClassifier(rf)

  # Addestro il modello sul train set di questa fold
  multi_output_xgb.fit(X_train, y_train)

  # Predico il target sul test set di questa fold
  y_pred = multi_output_xgb.predict(X_test)



  # Genero i classification report per ogni target
  target_names = ['PR_class', 'ER_class', 'KI67_class']
  reports = {}

  for i, target_name in enumerate(target_names):
      reports[target_name] = classification_report(
          y_test.iloc[:, i],
          y_pred[:, i],
          output_dict=True,  # Restituisce dizionario invece di stringa
          zero_division=0
      )

  return f1_score(y_test, y_pred, average="micro", zero_division=0), reports

# Training


In [ ]:
def training(file_path, csv_name):
    # Vado a leggere il csv
    df = pd.read_csv(file_path)

    # Definisco le colonne target
    original_target_list = ['PR [SII]', 'ER [SII]', 'KI67 [%]']

    # Vado a rimuovere le lesioni (righe) non valide
    df_validi = df.dropna(subset=original_target_list).copy()

    # Trasformo tutto in valori binari per "facilitare" il lavoro
    df_validi['PR_class'] = (df_validi['PR [SII]'] > 0.5).astype(int)
    df_validi['ER_class'] = (df_validi['ER [SII]'] > 0.5).astype(int)
    df_validi['KI67_class'] = (df_validi['KI67 [%]'] >= 20).astype(int)

    # Lista finale delle colonne target binarizzate che verranno usate per l'addestramento
    final_target_list = ['PR_class', 'ER_class', 'KI67_class']

    """
     Preparo le feature (X) e i target (y) per il modello
    """
    # Definisco tutte le colonne da rimuovere per ottenere solo le feature radiomiche
    features_to_drop = ['Patient ID', 'lesion idx', 'tumor/benign', 'GRADE', 'isTN', 'Breast'] + original_target_list + final_target_list
    features = df_validi.drop(columns=features_to_drop, errors='ignore')

    # 'target' contiene le 3 colonne da usare
    target = df_validi[final_target_list]

    # 'groups' contiene l'ID del paziente per ogni lesione.
    # Mi serve per fare la cross-validation a gruppo
    groups = df_validi['Patient ID']

    # Riempie a Nan se è rimasto vuoto
    features = features.fillna(features.mean())

    """ Dovrei pulire il nome delle colonne per farlo andare """
    features.columns = [re.sub(r'\[|\]|<', '', col) for col in features.columns]


    # Imposto la strategia di cross-validation.
    # GroupKFold assicura che le lesioni dello stesso paziente non vengano mai divise tra training set e test set
    cv = GroupKFold(n_splits=5, shuffle=True, random_state=42)

    # Lista vuota per collezionare i punteggi di performance di ogni fold.
    scores = []

    # Definisco gli iperparametri
    """iperparametri = {
        'n_estimators': [100, 200, 300],
        'max_depth': [3, 5, 7, 9],
        'learning_rate': [0.05, 1],
        'subsample': [0.8, 1.0, 2],
        'colsample_bytree': [0.8, 1],
        'min_child_weight': [1, 2, 3],
        'gamma': [0, 0.1, 2, 3],
        'reg_alpha': [0.1, 0.5, 1],
        'reg_lambda': [1, 2, 3]
    }"""

    iperparametri = {
        'n_estimators': [100],              # Numero di alberi
        'max_depth': [3, 5],                # Profondità massima degli alberi
        'learning_rate': [0.05],            # Tasso di apprendimento
        'subsample': [0.6, 1.0],            # Frazione di campioni per albero
        'colsample_bytree': [0.6],          # Frazione di feature per albero
        'min_child_weight': [1, 3],         # Somma minima dei pesi nelle foglie
        'gamma': [0.1],                     # Riduzione minima loss per split
        'reg_alpha': [0.1, 1],              # Regolarizzazione L1
        'reg_lambda': [1, 2]                # Regolarizzazione L2
    }

    scores = []

    # Calcolo il numero totale di combinazioni da testare per la barra di caricamento.
    total_combinations = math.prod(len(v) for v in iperparametri.values())

    # Dico come voglio vedere la barra
    inner_custom_format = "  {desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} [{elapsed}]"

    print(f"\nInizio Grid Search ({total_combinations} combinazioni) per: {csv_name}")


    combination_count = 0
    with tqdm.tqdm(total=total_combinations, desc="Combinazioni Testate",
                   bar_format="  {desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} [{elapsed}]",
                   leave=True, ncols=100) as pbar:
        for n_est in iperparametri['n_estimators']:
            for max_d in iperparametri['max_depth']:
                for lr in iperparametri['learning_rate']:
                    for sub in iperparametri['subsample']:
                        for col in iperparametri['colsample_bytree']:
                            for mcw in iperparametri['min_child_weight']:
                                for gam in iperparametri['gamma']:
                                    for alpha in iperparametri['reg_alpha']:
                                        for lam in iperparametri['reg_lambda']:
                                            combination_count += 1

                                            results = Parallel(n_jobs=-1)(
                                                delayed(fit_single_fold)(train_idx, test_idx, features, target, groups,
                                                                         n_est, max_d, lr, sub, col, mcw, gam, alpha, lam)
                                                for train_idx, test_idx in cv.split(features, target, groups)
                                            )

                                            fold_scores = [r[0] for r in results]
                                            fold_reports = [r[1] for r in results]
                                            mean_score = np.mean(fold_scores)
                                            std_score = np.std(fold_scores)

                                            pbar.set_postfix_str(f"F1: {mean_score:.3f} | LR: {lr} | MD: {max_d} | NE (Epoche): {n_est} | L1: {alpha}")
                                            pbar.update(1)

                                            scores.append({
                                                'n_estimators': n_est,
                                                'max_depth': max_d,
                                                'learning_rate': lr,
                                                'subsample': sub,
                                                'colsample_bytree': col,
                                                'min_child_weight': mcw,
                                                'gamma': gam,
                                                'reg_alpha': alpha,
                                                'reg_lambda': lam,
                                                'mean_score': mean_score,
                                                'std_score': std_score,
                                                'fold_scores': fold_scores,
                                                'fold_reports': fold_reports
                                            })

    return scores

# Vado a stampare il risultato in un formato leggibile

In [ ]:
def print_grid_search_results(results_per_dataset):
    """
    Stampa i risultati della Grid Search in modo organizzato e leggibile
    """
    print("\n" + "=" * 80)
    print(" " * 25 + "RIEPILOGO DEI MIGLIORI RISULTATI")
    print("=" * 80)
    
    # Lista per il riepilogo finale comparativo
    summary_data = []
    
    for name, metrics_list in results_per_dataset.items():
        best_result = max(metrics_list, key=lambda x: x['mean_score'])
        
        print(f"\n{'─' * 80}")
        print(f" Dataset: {name}")
        print(f"{'─' * 80}")
        print(f"\n Performance: F1-score = {best_result['mean_score']:.3f} ± {best_result['std_score']:.3f}\n")
        
        # Tabella Iperparametri
        print("Iperparametri Ottimali:")
        params_table = [
            ['n_estimators', best_result['n_estimators']],
            ['max_depth', best_result['max_depth']],
            ['learning_rate', best_result['learning_rate']],
            ['subsample', best_result['subsample']],
            ['colsample_bytree', best_result['colsample_bytree']],
            ['min_child_weight', best_result['min_child_weight']],
            ['gamma', best_result['gamma']],
            ['reg_alpha (L1)', best_result['reg_alpha']],
            ['reg_lambda (L2)', best_result['reg_lambda']]
        ]
        print(tabulate(params_table, headers=['Parametro', 'Valore'], tablefmt='simple'))
        
        # Metriche per target
        print("\n Metriche di Classificazione per Target:\n")
        target_names = ['PR_class', 'ER_class', 'KI67_class']
        
        for target_name in target_names:
            first_fold_report = best_result['fold_reports'][0][target_name]
            
            rows = []
            for cls in ['0', '1']:
                rows.append([
                    f"Classe {cls}",
                    f"{first_fold_report[cls]['precision']:.3f}",
                    f"{first_fold_report[cls]['recall']:.3f}",
                    f"{first_fold_report[cls]['f1-score']:.3f}",
                    int(first_fold_report[cls]['support'])
                ])
            
            print(f"  {target_name}:")
            print(tabulate(rows, headers=['', 'Precision', 'Recall', 'F1-score', 'Support'], 
                         tablefmt='simple', colalign=('left', 'center', 'center', 'center', 'center')))
            print()
        
        # Aggiungi al riepilogo comparativo
        summary_data.append([
            name,
            f"{best_result['mean_score']:.3f}",
            f"{best_result['std_score']:.3f}",
            best_result['max_depth'],
            best_result['learning_rate'],
            best_result['n_estimators']
        ])
    
    # Riepilogo Comparativo Finale
    print("\n" + "=" * 80)
    print(" " * 25 + "CONFRONTO TRA TUTTI I DATASET")
    print("=" * 80 + "\n")
    
    # Ordina per F1-score decrescente
    summary_data.sort(key=lambda x: float(x[1]), reverse=True)
    
    print(tabulate(summary_data, 
                   headers=['Dataset', 'F1-score', 'Std Dev', 'Max Depth', 'LR', 'N Est.'],
                   tablefmt='grid'))
    
    print("\n Analisi completata!\n")


# Lettura dei file

In [ ]:
# Esegui il training per tutti i dataset
results_per_dataset = {}
for name, file_path in datasets.items():
    results_per_dataset[name] = training(file_path, name)

# Usa la nuova funzione per stampare i risultati
print_grid_search_results(results_per_dataset)
